# Colab T4에서 YOLOv8n 실험 실행
공개 저장소를 다운로드하거나 필요한 폴더를 Drive에 놓은 뒤 사용한다. 기존 실험을 참고한 재실행용 노트북이며 과거 실행 출력은 포함하지 않는다.
런타임 유형을 T4 GPU로 선택한다. 학습할 때마다 Drive의 다른 실험 폴더에 저장한다.

In [ ]:
!nvidia-smi


In [ ]:
%pip install ultralytics==8.4.216 PyYAML matplotlib imageio-ffmpeg==0.8.1 lap==0.5.14


설치 후 런타임 재시작이 요구되면 재시작하고 다음 셀부터 순서대로 다시 실행한다. API 키는 이 노트북에 저장하지 않는다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


아래 경로를 실제 위치로 수정한다. 저장소 폴더에는 scripts/, models/, requirements.txt가 있어야 한다. 데이터 ZIP은 별도로 압축을 푼다.

In [ ]:
from pathlib import Path
import os, torch
REPO = Path("/content/drive/MyDrive/ML_desk_objects/repository")
DATA_V1 = Path("/content/Desk-Objects-Detection-1/data.yaml")
DATA_V2 = Path("/content/Desk-Objects-Detection-2/data.yaml")
OUTPUT = Path("/content/drive/MyDrive/ML_desk_objects/reproduced")
assert (REPO / "scripts/train.py").is_file(), "REPO 경로를 수정하세요"
assert DATA_V1.is_file() and DATA_V2.is_file(), "데이터를 내려받아 압축 해제하세요"
assert torch.cuda.is_available(), "T4 GPU 런타임을 선택하세요"
os.chdir(REPO)
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)
run("scripts/audit_dataset.py", "--data", DATA_V1, "--compare", DATA_V2, "--out", OUTPUT / "dataset_audit.json")


실험마다 yolov8n.pt에서 새로 시작한다. 아래 셀은 6번 학습하므로 오래 걸릴 수 있다. 필요한 한 조건만 실행하려면 반복 범위를 수정한다.

In [ ]:
for size, data in [(196, DATA_V1), (414, DATA_V2)]:
    for epochs in [25, 50, 75]:
        name = f"colab_{size}_{epochs}"
        run("scripts/train.py", "--data", data, "--epochs", epochs, "--device", 0, "--project", OUTPUT / "train", "--name", name)


실제 학습 출력 폴더를 확인하여 아래 RUN_DIR을 선택한다. 같은 이름을 다시 쓰면 접미사가 붙을 수 있다. 테스트를 확인하면서 학습 설정을 계속 조정하지 않는다.

In [ ]:
RUN_DIR = OUTPUT / "train/colab_414_50"
assert (RUN_DIR / "weights/best.pt").is_file()
run("scripts/evaluate.py", "--model", RUN_DIR / "weights/best.pt", "--data", DATA_V2, "--device", 0, "--project", OUTPUT / "evaluate", "--name", "colab_414_50")


완료 후 weights, results.csv, results.png, args.yaml, experiment.json을 보관한다. /content만 사용하면 런타임 종료 후 파일이 없어질 수 있다.